DSCI 552 Homework 6

Name: Brynn Dafoe GitHub Username: brynndafoe02 USD ID: 3109-6692-10

In [20]:
from imblearn.under_sampling import RandomUnderSampler
from itertools import combinations
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
import re

import seaborn as sns

from sklearn import datasets
from sklearn.datasets import make_classification, make_regression
from sklearn.decomposition import PCA
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression, LogisticRegressionCV, RidgeCV, LassoCV
from sklearn.metrics import RocCurveDisplay, roc_auc_score, accuracy_score, classification_report, confusion_matrix, mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV, train_test_split, StratifiedKFold, cross_val_score, RepeatedKFold, KFold
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.tree import DecisionTreeClassifier, plot_tree, _tree

from skmultilearn.problem_transform import LabelPowerset

import statsmodels.api as sm
import statsmodels.formula.api as smf

import xgboost as xgb

Question 1.b
- This data set has missing values
- When the number of data with missing values is significant, discarding them is not a good idea

In [21]:
# setting up dataframes, separating class labels from the predictors
training_df = pd.read_csv("../data/aps_failure_training_set.csv", skiprows=20, na_values = "na")
X_train_df = training_df.drop(columns=["class"])
Y_train_df = training_df["class"].map({"neg": 0, "pos": 1})

testing_df = pd.read_csv("../data/aps_failure_test_set.csv", skiprows=20, na_values = "na")
X_test_df = testing_df.drop(columns=["class"])
Y_test_df = testing_df["class"].map({"neg": 0, "pos": 1})


Question 1.b.i
- Research what types of techniques are usually used for dealing with data with missing values
- - They are called data imputation techniques
- Pick at least one of them and apply it to this data in the next steps
- - You are welcome to test more than one method

Common techniques:
- Mean Imputation (if data is distributed normally) -> 5-20% missing data, MCAR or MAR
- Median Imputation (if data is skewed) -> 5-20% missing data, MCAR or MAR
- Mode Imputation (if variables are categorical) -> 5-20% missing data, MCAR or MAR
- KNN Imputation (comp expensive for large datasets) -> more than 20%, MCAR or MAR
- Regression Imputation (when missing values can be predicted from other variables) -> 5-20% missing data, MCAR or MAR
- MICE (computationally intense) -> more than 20%, MCAR, MAR, or NMAR
- Random Sampling Imputation -> 5-20% missing data, MCAR or MAR

In [22]:
print("Number of missing data in entire training set:")
print(training_df.isna().sum().sum())
print("Percentage of missing data relative to entire set:")
print((training_df.isna().sum().sum())/(60000*171))

Number of missing data in entire training set:
850015
Percentage of missing data relative to entire set:
0.08284746588693957


In [23]:
with pd.option_context('display.max_rows', None):
    print(X_train_df.skew().sort_values(ascending=False))


cs_009    243.531177
cf_000    212.459407
co_000    212.459406
ad_000    212.459406
dh_000    202.907014
az_009    163.609146
ay_009    159.066837
ag_000    153.387555
dj_000    150.507647
cs_007    140.732294
cs_008    121.563916
ak_000    113.625497
as_000    111.732428
au_000    108.987809
df_000    108.938646
ct_000    107.031061
ay_001    102.415758
cn_009    101.075836
ag_009     98.288989
dz_000     92.410273
ea_000     91.230552
ae_000     90.588252
eg_000     89.029482
az_002     86.763535
db_000     83.627961
dg_000     76.812727
ag_001     76.076049
aj_000     75.075731
ef_000     73.432440
az_000     69.347485
cs_001     67.935911
ch_000     67.165468
cy_000     65.341916
ay_004     63.509322
az_007     63.194700
da_000     62.925455
az_008     62.461056
ay_000     59.621486
az_001     58.303989
at_000     57.316795
cn_000     56.637624
cz_000     55.757088
ay_002     53.640077
dl_000     53.369202
ba_008     53.345250
af_000     51.671770
ba_009     49.209881
dk_000     49

The training data set proves to be highly skewed to the right, therefore I will use median imputation because of the skewness and because the missing data is 8%.